# Лабораторная работа №2. Логистическая и линейная регрессия

Фокин Леонид Александрович, группа 401, вариант 20.

## Данные и метрики

Используются Wholesale Customers (`Channel`, macro-F1 и accuracy) и SGEMM (время ядра в мс, MAE и RMSE). Тестовая доля 20%, пять фолдов для подбора, `random_state=20`. Подробнее о наборах: [data/README.md](data/README.md). При подборе максимизирую macro-F1 и минимизирую MAE.

Загружаю общие функции разбиения и оценки. В этой работе библиотечный бейзлайн использует стандартизацию признаков.

In [1]:
from sklearn.linear_model import LogisticRegression, LinearRegression, Ridge
import numpy as np
import pandas as pd
from IPython.display import display
from study import SEED, TASKS, evaluate, split, validate

BASE = 'scale'

def sklearn_model(task, params):
    if task == 'classification':
        return LogisticRegression(max_iter=2000, **params)
    return Ridge(**params) if params else LinearRegression()

## Бейзлайн sklearn

Фиксирую качество моделей со стандартными параметрами. Тестовые оценки не участвуют в выборе конфигурации.

In [2]:
baseline = {task: evaluate(task, sklearn_model(task, {}), BASE) for task in TASKS}
display(pd.DataFrame([metrics for metrics, _ in baseline.values()]).round(4))

,task,f1_macro,accuracy,fit_seconds,mae_ms,rmse_ms
0,classification,0.8832,0.8977,0.0028,NaN,NaN
1,regression,NaN,NaN,0.0020,178.7355,266.6248


## Гипотезы и кросс-валидация

Стандартизация включена уже в бейзлайне. Логарифм трат и доли категорий могут упростить границу между каналами продаж. В регрессии кратности плитки и условные признаки кэша дают линейной модели дополнительные объясняющие переменные. Проверяю штраф L2 отдельно: меньший C означает более сильный штраф в классификации, больший alpha означает более сильный штраф в регрессии. При `alpha > 0` использую `Ridge` вместо `LinearRegression`.

На пяти фолдах сравниваю три варианта: новые признаки, подбор параметров и оба изменения вместе. `cv_std` показывает разброс оценок по фолдам, а не доверительный интервал.

In [3]:
feature_options = ['features_scale']
class_grid = [{'C': 0.03}, {'C': 0.1}, {'C': 0.3}, {'C': 3}, {'C': 10}, {'C': 30}]
reg_grid = [{'alpha': 0.1}, {'alpha': 1}, {'alpha': 10}, {'alpha': 50}, {'alpha': 200}]
tables, chosen = {}, {}
for task, grid in zip(TASKS, (class_grid, reg_grid)):
    candidates = [('baseline', BASE, {})]
    candidates += [('preprocessing', prep, {}) for prep in feature_options]
    candidates += [('parameters', BASE, params) for params in grid]
    candidates += [('combined', prep, params) for prep in feature_options for params in grid]
    table = tables[task] = validate(task, lambda params: sklearn_model(task, params), candidates)
    chosen[task] = table.loc[table.experiment != 'baseline'].iloc[0]
    print(task)
    display(table.drop_duplicates('experiment').round(4).reset_index(drop=True))

classification


,experiment,prep,params,cv_mean,cv_std
0,parameters,scale,{'C': 3},0.9010,0.0383
1,baseline,scale,{},0.8975,0.0359
2,combined,features_scale,{'C': 0.03},0.8959,0.0481
3,preprocessing,features_scale,{},0.8799,0.0384


regression


,experiment,prep,params,cv_mean,cv_std
0,parameters,scale,{'alpha': 200},170.5096,12.5561
1,baseline,scale,{},178.3097,11.4854
2,combined,features_scale,{'alpha': 200},181.2115,12.8655
3,preprocessing,features_scale,{},184.1464,11.6312


Обучаю лучшую по кросс-валидации альтернативу на всей обучающей выборке и проверяю её на тесте.

In [4]:
improved = {task: evaluate(task, sklearn_model(task, row.params), row.prep)
            for task, row in chosen.items()}
display(pd.DataFrame([metrics for metrics, _ in improved.values()]).round(4))

,task,f1_macro,accuracy,fit_seconds,mae_ms,rmse_ms
0,classification,0.8971,0.9091,0.0032,NaN,NaN
1,regression,NaN,NaN,0.0026,170.3948,263.8061


## Самописная реализация

Логистическая модель минимизирует сумму логистических потерь с L2-штрафом. Свободный член не штрафуется. Использую шаг Ньютона из градиента и гессиана с уменьшением шага при росте потерь. Линейная модель решает МНК через `lstsq`; Ridge добавляет к системе строки для штрафа на коэффициенты. Метод оптимизации не делегируется sklearn.

Для самописной модели беру те же признаки и параметры, что для sklearn: сначала исходные, потом выбранные по кросс-валидации.

In [5]:
from models.linear import Logistic, LeastSquares

def own_model(task, params):
    return (Logistic if task == "classification" else LeastSquares)(**params)

own_base = {task: evaluate(task, own_model(task, {}), BASE) for task in TASKS}
own_improved = {task: evaluate(task, own_model(task, row.params), row.prep)
                for task, row in chosen.items()}

Для каждой задачи сохраняю результаты четырёх версий. `Δ` показывает разницу метрик, а не процентное изменение. Для MAE отрицательное значение означает меньшую ошибку.

In [6]:
versions = {'sklearn baseline': baseline, 'sklearn improved': improved,
            'own baseline': own_base, 'own improved': own_improved}
scores = pd.DataFrame([dict(version=version, **metrics)
                       for version, outcomes in versions.items()
                       for metrics, _ in outcomes.values()])
from pathlib import Path

Path('results').mkdir(exist_ok=True)
scores.drop(columns='fit_seconds').to_csv('results/lab2.csv', index=False)
for task, metric in zip(TASKS, ('f1_macro', 'mae_ms')):
    subset = scores.query('task == @task').dropna(axis=1).set_index('version')
    display(subset.drop(columns='task').round(4))
    s = subset[metric]
    display(pd.Series({'sklearn improved − sklearn baseline': s.iloc[1] - s.iloc[0],
                       'own baseline − sklearn baseline': s.iloc[2] - s.iloc[0],
                       'own improved − sklearn improved': s.iloc[3] - s.iloc[1],
                       'own improved − own baseline': s.iloc[3] - s.iloc[2]},
                      name=f'Δ {metric}').round(4))

,f1_macro,accuracy,fit_seconds
version,,,
sklearn baseline,0.8832,0.8977,0.0028
sklearn improved,0.8971,0.9091,0.0032
own baseline,0.8832,0.8977,0.0016
own improved,0.8971,0.9091,0.0025


sklearn improved − sklearn baseline    0.0139
own baseline − sklearn baseline        0.0000
own improved − sklearn improved        0.0000
own improved − own baseline            0.0139
Name: Δ f1_macro, dtype: float64

,fit_seconds,mae_ms,rmse_ms
version,,,
sklearn baseline,0.0020,178.7355,266.6248
sklearn improved,0.0026,170.3948,263.8061
own baseline,0.0027,178.7355,266.6248
own improved,0.0024,170.3948,263.8061


sklearn improved − sklearn baseline   -8.3407
own baseline − sklearn baseline        0.0000
own improved − sklearn improved       -0.0000
own improved − own baseline           -8.3407
Name: Δ mae_ms, dtype: float64

Сравниваю матрицы ошибок обеих настроенных моделей. Для регрессии считаю MAE отдельно на быстрых и медленных конфигурациях; порог деления беру по медиане обучающей выборки.

In [7]:
from sklearn.metrics import confusion_matrix

_, _, _, test_channel = split('classification')
_, _, train_time, test_time = split('regression')
bands = np.where(test_time <= train_time.median(), 'быстрые', 'медленные')
for label, outcomes in [('sklearn', improved), ('самописная', own_improved)]:
    print(label)
    display(pd.DataFrame(confusion_matrix(test_channel, outcomes['classification'][1], labels=[1, 2]),
                         index=['факт Horeca', 'факт Retail'],
                         columns=['прогноз Horeca', 'прогноз Retail']))
    error = np.abs(np.asarray(test_time) - outcomes['regression'][1])
    display(pd.Series(error).groupby(bands).agg(['size', 'mean']).round(2))

sklearn


,прогноз Horeca,прогноз Retail
факт Horeca,55,5
факт Retail,3,25


,size,mean
быстрые,308,126.58
медленные,292,216.61


самописная


,прогноз Horeca,прогноз Retail
факт Horeca,55,5
факт Retail,3,25


,size,mean
быстрые,308,126.58
медленные,292,216.61


## Выводы

- **Классификация.** На CV выиграл `C=3` со стандартными признаками: macro-F1 0,9010 против 0,8975 у бейзлайна. Логарифмы и доли в этом эксперименте пользы не дали. На тесте macro-F1 вырос с 0,8832 до 0,8971. Самописная оптимизация дала те же тестовые оценки для обоих рецептов.
- **Регрессия.** Ridge с `alpha=200` снизила CV MAE с 178,31 до 170,51 мс, а тестовую MAE с 178,74 до 170,39 мс. Добавление взаимодействий не улучшило CV. Самописные МНК и Ridge совпали со sklearn по метрикам с точностью до округления. У обеих версий линейной модели ошибка заметно больше, чем у KNN: одной глобальной суммы эффектов недостаточно, чтобы точно описать измеренные времена.